# Results: Semantic Profile Analysis of Visualization Comments

Five research questions:

1. **Cross-graph consistency** — How consistently do people go beyond the plot across different graphs?
2. **Within-plot commenter differences** — What category accounts for the largest gap between long vs short comments?
3. **Within-comment positioning** — When does beyond-plot content appear? Early, late, intermixed?
4. **Network characterization** — Density, centrality, edge count, and which nodes receive the most incoming edges.
5. **Sequence statistics** — What are the most common category transition chains?

In [34]:
from __future__ import annotations

import json
import warnings
from collections import Counter, defaultdict
from itertools import product
from pathlib import Path
from typing import Dict, List, Tuple

import altair as alt
import networkx as nx
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
alt.data_transformers.enable("vegafusion")

# ── Load all four articles ──────────────────────────────────────────────
DATA_DIR = Path("..") / "combined_data"
ARTICLE_IDS = ["91", "173", "181", "183", "35"]

all_comments: List[dict] = []
for aid in ARTICLE_IDS:
    p = DATA_DIR / f"{aid}.json"
    if p.exists():
        with p.open() as f:
            all_comments.extend(json.load(f))

print(f"Loaded {len(all_comments)} comments across {len(ARTICLE_IDS)} articles")

Loaded 895 comments across 5 articles


In [35]:
# ── Functional group mapping ────────────────────────────────────────────
TAG_TO_GROUP: Dict[str, str] = {
    "Visual feature detection":             "visual_observation",
    "Visual data extraction":               "visual_observation",
    "Encoding interpretation":              "visual_observation",
    "L1: Elemental and encoded properties": "visual_observation",
    "L2: Statistical concepts and relations": "visual_observation",
    "L3: Trend and pattern analysis":       "visual_observation",
    "Background knowledge":                 "background_knowledge",
    "Explanatory inference":                "hypothetical_reasoning",
    "Predictive / counterfactual inference": "hypothetical_reasoning",
    "Information need / curiosity":         "hypothetical_reasoning",
    "Evaluative / affective judgment":      "personal_relevance",
    "Personal/episodic retrieval":          "personal_relevance",
    "Meta / paratext":                      "meta",
}
GROUPS = ["visual_observation", "background_knowledge", "hypothetical_reasoning", "personal_relevance", "meta"]

BEYOND_PLOT_TAGS = {
    "Background knowledge",
    "Explanatory inference",
    "Predictive / counterfactual inference",
    "Information need / curiosity",
    "Personal/episodic retrieval",
    "Evaluative / affective judgment",
}

def to_group(tag: str | None) -> str:
    if not tag:
        return "unknown"
    return TAG_TO_GROUP.get(tag, "unknown")

# ── Build flat node & edge tables ───────────────────────────────────────
node_rows: list[dict] = []
edge_rows: list[dict] = []

for comment in all_comments:
    aid = comment["article_id"]
    ci = comment["comment_index"]
    dg = comment.get("dependency_graph") or []
    n_nodes = len(dg)
    for node in dg:
        nid = node["id"]
        tag = node.get("comment_tag", "unknown")
        group = to_group(tag)
        in_deg = 0
        for other in dg:
            for dep in other.get("depends_on", []):
                if dep["id"] == nid:
                    in_deg += 1
        out_deg = len(node.get("depends_on", []))
        node_rows.append({
            "article_id": aid,
            "comment_index": ci,
            "node_id": nid,
            "sentence": node["sentence"],
            "tag": tag,
            "group": group,
            "in_degree": in_deg,
            "out_degree": out_deg,
            "position": nid,
            "n_nodes": n_nodes,
        })
        for dep in node.get("depends_on", []):
            edge_rows.append({
                "article_id": aid,
                "comment_index": ci,
                "source": dep["id"],
                "target": nid,
                "edge_type": dep.get("edge_type", "unknown"),
            })

nodes = pd.DataFrame(node_rows)
edges = pd.DataFrame(edge_rows)

# Normalised position: 0 = first sentence, 1 = last sentence
nodes["rel_position"] = nodes["position"] / nodes["n_nodes"].clip(lower=1)

print(f"Nodes: {len(nodes):,}   Edges: {len(edges):,}")
nodes.head(3)

Nodes: 12,023   Edges: 12,584


,article_id,comment_index,node_id,sentence,tag,group,in_degree,out_degree,position,n_nodes,rel_position
0,91,1,0,I notice a graph.,Visual feature detection,visual_observation,2,0,0,11,0.000000
1,91,1,1,The graph is shown about diversity in the athl...,Meta / paratext,meta,7,1,1,11,0.090909
2,91,1,2,The graph shows a percentile.,Encoding interpretation,visual_observation,3,1,2,11,0.181818


## Q1 — Cross-graph consistency of beyond-plot content

How consistently do people bring in semantic knowledge, personal experience, hypotheticals, and curiosity across different graphs?

We define **"beyond-plot"** broadly as everything outside L1–L3 reading of the chart: Background knowledge, Explanatory inference, Predictive/counterfactual inference, Information need / curiosity, Personal/episodic retrieval, and Evaluative/affective judgment.

In [36]:
# ── Q1a: Per-article distribution of functional groups ──────────────────
group_counts = (
    nodes.groupby(["article_id", "group"])
    .size()
    .reset_index(name="count")
)
group_totals = group_counts.groupby("article_id")["count"].transform("sum")
group_counts["proportion"] = group_counts["count"] / group_totals

group_order = ["visual_observation", "background_knowledge", "hypothetical_reasoning", "personal_relevance", "meta", "unknown"]

chart_q1a = (
    alt.Chart(group_counts, title="Functional-group proportions by article")
    .mark_bar()
    .encode(
        x=alt.X("article_id:N", title="Article", sort=ARTICLE_IDS),
        y=alt.Y("proportion:Q", title="Proportion of sentences", stack="normalize"),
        color=alt.Color(
            "group:N",
            title="Group",
            sort=group_order,
            scale=alt.Scale(
                domain=group_order,
                range=["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#bab0ac"],
            ),
        ),
        tooltip=["article_id:N", "group:N", "count:Q", alt.Tooltip("proportion:Q", format=".1%")],
    )
    .properties(width=400, height=350)
)
chart_q1a

alt.Chart(...)

In [41]:
# ── Q1b: Per-comment beyond-plot fraction, by article (strip/box/point mean+boot ci) ───────

import numpy as np

def bootstrap_mean_ci(data, n_bootstrap=2000, ci=95, random_state=None):
    """
    Compute the bootstrap mean and confidence interval for an array.
    Returns (mean, lower, upper).
    """
    rng = np.random.default_rng(random_state)
    means = []
    data = np.array(data)
    for _ in range(n_bootstrap):
        sample = rng.choice(data, size=len(data), replace=True)
        means.append(np.mean(sample))
    m = np.mean(means)
    lo = np.percentile(means, (100 - ci) / 2)
    hi = np.percentile(means, 100 - (100 - ci) / 2)
    return m, lo, hi

# Collect bootstrapped means/CIs by article_id
boot_stats = []
for aid, grp in comment_stats.groupby("article_id"):
    vals = grp["beyond_frac"].values
    m, lo, hi = bootstrap_mean_ci(vals)
    boot_stats.append(dict(article_id=aid, mean=m, ci_lo=lo, ci_hi=hi))
bootstats_df = pd.DataFrame(boot_stats)

# Point plot for mean, with error bars for bootstrap CI
mean_pts = (
    alt.Chart(bootstats_df)
    .mark_point(color="black", filled=True, size=80)
    .encode(
        x=alt.X("article_id:N", title="Article", sort=ARTICLE_IDS),
        y=alt.Y("mean:Q", title="Mean beyond-plot fraction", scale=alt.Scale(domain=[0,1])),
        tooltip=[alt.Tooltip("mean:Q", format=".1%"), 
                 alt.Tooltip("ci_lo:Q", format=".1%"), 
                 alt.Tooltip("ci_hi:Q", format=".1%")]
    )
)
error_bars = (
    alt.Chart(bootstats_df)
    .mark_errorbar()
    .encode(
        x=alt.X("article_id:N", sort=ARTICLE_IDS),
        y=alt.Y("ci_lo:Q"),
        y2=alt.Y2("ci_hi:Q")
    )
)

strip = (
    alt.Chart(comment_stats)
    .mark_circle(size=30, opacity=0.35)
    .encode(
        x=alt.X("article_id:N", title="Article", sort=ARTICLE_IDS),
        y=alt.Y("beyond_frac:Q", title="Fraction beyond-plot per comment"),
        tooltip=["article_id:N", "comment_index:Q", "n_sentences:Q", "n_beyond:Q",
                 alt.Tooltip("beyond_frac:Q", format=".0%")],
    )
)
box = (
    alt.Chart(comment_stats)
    .mark_boxplot(extent="min-max", size=30, median={"color": "black"})
    .encode(
        x=alt.X("article_id:N", title="Article", sort=ARTICLE_IDS),
        y=alt.Y("beyond_frac:Q", title=""),
    )
)
chart_q1b = (
    (error_bars + mean_pts)
    .properties(width=100, height=350, title="Per-comment beyond-plot fraction by article")
)
chart_q1b

alt.LayerChart(...)

In [6]:
# ── Q1c: Fine-grained beyond-plot tag breakdown by article ──────────────
beyond_nodes = nodes[nodes["tag"].isin(BEYOND_PLOT_TAGS)]
beyond_counts = (
    beyond_nodes.groupby(["article_id", "tag"])
    .size()
    .reset_index(name="count")
)
beyond_totals = beyond_counts.groupby("article_id")["count"].transform("sum")
beyond_counts["proportion"] = beyond_counts["count"] / beyond_totals

chart_q1c = (
    alt.Chart(beyond_counts, title="Beyond-plot sub-category mix by article")
    .mark_bar()
    .encode(
        x=alt.X("article_id:N", title="Article", sort=ARTICLE_IDS),
        y=alt.Y("proportion:Q", title="Proportion within beyond-plot", stack="normalize"),
        color=alt.Color("tag:N", title="Category"),
        tooltip=["article_id:N", "tag:N", "count:Q", alt.Tooltip("proportion:Q", format=".1%")],
    )
    .properties(width=400, height=350)
)
chart_q1c

alt.Chart(...)

In [7]:
# ── Q1d: Consistency summary table ──────────────────────────────────────
q1_summary = (
    comment_stats.groupby("article_id")["beyond_frac"]
    .agg(["mean", "std", "median", "min", "max", "count"])
    .rename(columns={"count": "n_comments"})
    .round(3)
)
overall_mean = comment_stats["beyond_frac"].mean()
overall_std = comment_stats["beyond_frac"].std()
print(f"Overall beyond-plot fraction:  mean={overall_mean:.3f}  std={overall_std:.3f}")
print()
q1_summary

Overall beyond-plot fraction:  mean=0.566  std=0.201



,mean,std,median,min,max,n_comments
article_id,,,,,,
173,0.635,0.201,0.667,0.0,1.0,241
181,0.509,0.174,0.500,0.0,1.0,153
183,0.572,0.227,0.600,0.0,1.0,137
91,0.524,0.179,0.533,0.0,1.0,219


## Q2 — Within-plot commenter differences: long vs short comments

For each article, split comments into terciles by length (sentence count). Compare the **semantic profile** (distribution of categories) of the longest 33% vs shortest 33%, and identify the category that accounts for the largest difference.

In [8]:
# ── Q2a: Compute tercile membership ─────────────────────────────────────
comment_len = (
    nodes.groupby(["article_id", "comment_index"])
    .size()
    .reset_index(name="n_sentences")
)

tercile_rows = []
for aid, grp in comment_len.groupby("article_id"):
    q33 = grp["n_sentences"].quantile(0.333)
    q66 = grp["n_sentences"].quantile(0.667)
    for _, row in grp.iterrows():
        if row["n_sentences"] <= q33:
            t = "short (bottom 33%)"
        elif row["n_sentences"] >= q66:
            t = "long (top 33%)"
        else:
            t = "middle"
        tercile_rows.append({
            "article_id": row["article_id"],
            "comment_index": row["comment_index"],
            "n_sentences": row["n_sentences"],
            "tercile": t,
        })

comment_len = pd.DataFrame(tercile_rows)
nodes_t = nodes.merge(comment_len, on=["article_id", "comment_index"])

extremes = nodes_t[nodes_t["tercile"].isin(["short (bottom 33%)", "long (top 33%)"])]
print("Comments per tercile per article:")
print(comment_len[comment_len["tercile"] != "middle"].groupby(["article_id", "tercile"]).size().unstack(fill_value=0))
print()
print(f"Sentences in short: {len(extremes[extremes['tercile']=='short (bottom 33%)'])}  "
      f"long: {len(extremes[extremes['tercile']=='long (top 33%)'])}")

Comments per tercile per article:
tercile     long (top 33%)  short (bottom 33%)
article_id                                    
173                     90                  90
181                     52                  57
183                     50                  48
91                      78                  89

Sentences in short: 2336  long: 5827


In [9]:
# ── Q2b: Semantic profiles of long vs short (pooled across articles) ────
ALL_TAGS = sorted(nodes["tag"].unique())

def profile(df, tag_col="tag"):
    counts = df[tag_col].value_counts()
    return (counts / counts.sum()).reindex(ALL_TAGS, fill_value=0.0)

prof_short = profile(extremes[extremes["tercile"] == "short (bottom 33%)"])
prof_long  = profile(extremes[extremes["tercile"] == "long (top 33%)"])
diff = prof_long - prof_short

profile_df = pd.DataFrame({
    "tag": ALL_TAGS,
    "short": prof_short.values,
    "long": prof_long.values,
    "diff (long − short)": diff.values,
}).sort_values("diff (long − short)", key=abs, ascending=False)

print("Category with LARGEST absolute difference between long and short comments:")
top_row = profile_df.iloc[0]
print(f"  {top_row['tag']}   Δ = {top_row['diff (long − short)']:+.3f}  "
      f"(short={top_row['short']:.3f}  long={top_row['long']:.3f})")
print()
profile_df

Category with LARGEST absolute difference between long and short comments:
  Information need / curiosity   Δ = -0.047  (short=0.149  long=0.102)



,tag,short,long,diff (long − short)
4,Information need / curiosity,0.149401,0.102454,-0.046947
0,Background knowledge,0.098459,0.141582,0.043123
6,Personal/episodic retrieval,0.139127,0.110692,-0.028435
3,Explanatory inference,0.065497,0.090098,0.024601
8,Uncategorizable,0.037243,0.019564,-0.017679
9,Visual data extraction,0.237158,0.253132,0.015974
2,Evaluative / affective judgment,0.068065,0.078428,0.010363
5,Meta / paratext,0.127140,0.119444,-0.007696
1,Encoding interpretation,0.022688,0.025571,0.002882
10,Visual feature detection,0.010274,0.012700,0.002426


In [10]:
# ── Q2c: Diverging bar chart of long−short differences ──────────────────
diff_df = profile_df[["tag", "diff (long − short)"]].copy()
diff_df.columns = ["tag", "diff"]
diff_df["abs_diff"] = diff_df["diff"].abs()
diff_df = diff_df.sort_values("abs_diff", ascending=True)

chart_q2c = (
    alt.Chart(diff_df, title="Semantic profile difference: long vs short comments")
    .mark_bar()
    .encode(
        x=alt.X("diff:Q", title="Δ proportion (long − short)"),
        y=alt.Y("tag:N", sort=alt.EncodingSortField(field="abs_diff", order="descending"), title=""),
        color=alt.condition(
            alt.datum.diff > 0,
            alt.value("#f28e2b"),
            alt.value("#4e79a7"),
        ),
        tooltip=["tag:N", alt.Tooltip("diff:Q", format="+.3f")],
    )
    .properties(width=500, height=380)
)

rule = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(strokeDash=[4, 4], color="gray").encode(x="x:Q")

chart_q2c + rule

alt.LayerChart(...)

In [49]:
# ── Q2c: Diverging bar chart, summed across category groups ─────────────

diff_df = profile_df[["tag", "diff (long − short)"]].copy()
diff_df.columns = ["tag", "diff"]

# Map categories to the (corrected) group labels from TAG_TO_GROUP
diff_df["group"] = diff_df["tag"].map(lambda t: TAG_TO_GROUP.get(t, "Unknown"))
diff_df["group"] = diff_df["group"].replace("categoreis", "categories", regex=True)

# Sum the differences and take sum of absolute differences for each group
group_sums = (
    diff_df
    .groupby("group", as_index=False)
    .agg(
        diff_sum=("diff", "sum"),
        abs_diff_sum=("diff", lambda x: abs(x).sum())
    )
)

# Sort by absolute summed difference so largest overall group diff is on top
group_sums = group_sums.sort_values("abs_diff_sum", ascending=True)

chart_q2c_group = (
    alt.Chart(group_sums, title="Summed semantic profile difference: long vs short (by category group)")
    .mark_bar()
    .encode(
        x=alt.X("diff_sum:Q", title="Σ Δ proportion (long − short)"),
        y=alt.Y("group:N", sort=alt.EncodingSortField(field="abs_diff_sum", order="descending"), title="Category group"),
        color=alt.condition(
            alt.datum.diff_sum > 0,
            alt.value("#f28e2b"),
            alt.value("#4e79a7"),
        ),
        tooltip=[
            alt.Tooltip("group:N", title="Group"),
            alt.Tooltip("diff_sum:Q", title="Sum Δ", format="+.3f"),
            alt.Tooltip("abs_diff_sum:Q", title="Sum |Δ|", format=".3f")
        ],
    )
    .properties(width=500, height=380)
)

rule = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(strokeDash=[4, 4], color="gray").encode(x="x:Q")

chart_q2c_group + rule

alt.LayerChart(...)

In [11]:
# ── Q2d: Faceted by article ──────────────────────────────────────────────
rows = []
for aid in ARTICLE_IDS:
    sub = extremes[extremes["article_id"] == aid]
    if sub.empty:
        continue
    ps = profile(sub[sub["tercile"] == "short (bottom 33%)"])
    pl = profile(sub[sub["tercile"] == "long (top 33%)"])
    d = pl - ps
    for tag in ALL_TAGS:
        rows.append({"article_id": aid, "tag": tag, "diff": d[tag]})

diff_by_article = pd.DataFrame(rows)
diff_by_article["abs_diff"] = diff_by_article["diff"].abs()

chart_q2d = (
    alt.Chart(diff_by_article, title="Long−short profile difference per article")
    .mark_bar()
    .encode(
        x=alt.X("diff:Q", title="Δ proportion"),
        y=alt.Y("tag:N", sort=alt.EncodingSortField(field="abs_diff", order="descending"), title=""),
        color=alt.condition(alt.datum.diff > 0, alt.value("#f28e2b"), alt.value("#4e79a7")),
        tooltip=["article_id:N", "tag:N", alt.Tooltip("diff:Q", format="+.3f")],
    )
    .facet(facet=alt.Facet("article_id:N", title="Article", sort=ARTICLE_IDS), columns=2)
    .properties(title="")
    .resolve_scale(y="shared")
)
chart_q2d

alt.FacetChart(...)

In [42]:
# ── Q2d: Faceted by category (TAG_TO_GROUP) ──────────────────────────────
rows = []
for aid in ARTICLE_IDS:
    sub = extremes[extremes["article_id"] == aid]
    if sub.empty:
        continue
    ps = profile(sub[sub["tercile"] == "short (bottom 33%)"])
    pl = profile(sub[sub["tercile"] == "long (top 33%)"])
    d = pl - ps
    for tag in ALL_TAGS:
        rows.append({
            "article_id": aid,
            "tag": tag,
            "diff": d[tag],
            "group": TAG_TO_GROUP.get(tag, "unknown")
        })

diff_by_article = pd.DataFrame(rows)
diff_by_article["abs_diff"] = diff_by_article["diff"].abs()

chart_q2d_category = (
    alt.Chart(diff_by_article, title="Long−short profile difference per category")
    .mark_bar()
    .encode(
        x=alt.X("diff:Q", title="Δ proportion"),
        y=alt.Y("tag:N", sort=alt.EncodingSortField(field="abs_diff", order="descending"), title=""),
        color=alt.condition(alt.datum.diff > 0, alt.value("#f28e2b"), alt.value("#4e79a7")),
        tooltip=["article_id:N", "tag:N", "group:N", alt.Tooltip("diff:Q", format="+.3f")],
    )
    .facet(facet=alt.Facet("group:N", title="Category group"), columns=2)
    .properties(title="")
    .resolve_scale(y="shared")
)
chart_q2d_category

alt.FacetChart(...)

## Q3 — Within-comment positioning: when does beyond-plot content appear?

For each sentence in a comment, we track its **normalised position** (0 = first, 1 = last). Is beyond-plot content front-loaded, back-loaded, or uniformly intermixed?

In [51]:
# ── Q3a: Density of relative position by group ─────────────────────────
# Only keep comments with ≥ 3 sentences so position is meaningful
nodes_q3 = nodes[nodes["n_nodes"] >= 3].copy()

density_data = nodes_q3[["group", "rel_position"]].copy()
density_data = density_data[density_data["group"] != "unknown"]

chart_q3a = (
    alt.Chart(density_data, title="Where in the comment does each group appear?")
    .transform_density(
        "rel_position",
        as_=["rel_position", "density"],
        groupby=["group"],
        extent=[0, 1],
        bandwidth=0.06,
    )
    .mark_area(opacity=0.55, interpolate="monotone")
    .encode(
        x=alt.X("rel_position:Q", title="Normalised position in comment (0=start, 1=end)"),
        y=alt.Y("density:Q", title="Density", stack=None),
        color=alt.Color(
            "group:N",
            title="Group",
            sort=GROUPS,
            scale=alt.Scale(
                domain=GROUPS,
                range=["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"],
            ),
        ),
    )
    .properties(width=600, height=300)
)
chart_q3a

alt.Chart(...)

In [13]:
# ── Q3b: Mean position per fine-grained beyond-plot tag ─────────────────
beyond_pos = nodes_q3[nodes_q3["tag"].isin(BEYOND_PLOT_TAGS)].copy()

mean_pos = (
    beyond_pos.groupby("tag")["rel_position"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values("mean")
)
mean_pos["se"] = mean_pos["std"] / np.sqrt(mean_pos["count"])
mean_pos["lo"] = mean_pos["mean"] - 1.96 * mean_pos["se"]
mean_pos["hi"] = mean_pos["mean"] + 1.96 * mean_pos["se"]

points = (
    alt.Chart(mean_pos)
    .mark_point(filled=True, size=80)
    .encode(
        x=alt.X("mean:Q", title="Mean normalised position (95% CI)", scale=alt.Scale(domain=[0, 1])),
        y=alt.Y("tag:N", sort=alt.EncodingSortField(field="mean", order="ascending"), title=""),
    )
)
errorbars = (
    alt.Chart(mean_pos)
    .mark_errorbar()
    .encode(
        x=alt.X("lo:Q", title=""),
        x2="hi:Q",
        y=alt.Y("tag:N", sort=alt.EncodingSortField(field="mean", order="ascending")),
    )
)
ref_line = (
    alt.Chart(pd.DataFrame({"x": [0.5]}))
    .mark_rule(strokeDash=[4, 4], color="gray")
    .encode(x="x:Q")
)

chart_q3b = (
    (ref_line + errorbars + points)
    .properties(width=500, height=250, title="Mean position of beyond-plot categories within comments")
)
chart_q3b

alt.LayerChart(...)

In [14]:
# ── Q3c: Heatmap — position (binned) × group ───────────────────────────
nodes_q3c = nodes_q3[nodes_q3["group"] != "unknown"].copy()
nodes_q3c["pos_bin"] = pd.cut(
    nodes_q3c["rel_position"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=["0.0–0.2", "0.2–0.4", "0.4–0.6", "0.6–0.8", "0.8–1.0"],
    include_lowest=True,
)

heat_data = (
    nodes_q3c.groupby(["pos_bin", "group"])
    .size()
    .reset_index(name="count")
)
heat_totals = heat_data.groupby("pos_bin")["count"].transform("sum")
heat_data["proportion"] = heat_data["count"] / heat_totals

chart_q3c = (
    alt.Chart(heat_data, title="Group share by position quintile")
    .mark_rect()
    .encode(
        x=alt.X("pos_bin:O", title="Position in comment"),
        y=alt.Y("group:N", title="", sort=GROUPS),
        color=alt.Color("proportion:Q", title="Proportion", scale=alt.Scale(scheme="blues")),
        tooltip=["pos_bin:O", "group:N", "count:Q", alt.Tooltip("proportion:Q", format=".1%")],
    )
    .properties(width=400, height=200)
)
chart_q3c

alt.Chart(...)

## Q4 — Network characterization

Each comment's dependency graph is a directed network. We compute:
- **Density** (edges / possible edges)
- **Average in-degree and out-degree** by category
- **PageRank centrality** — which nodes concentrate importance?
- **"Business end of the arrow"** — which categories receive the most incoming edges (i.e. are the *targets* that other statements build toward)?

In [15]:
# ── Q4a: Build NetworkX graphs per comment, compute metrics ─────────────
graph_metrics: list[dict] = []

for comment in all_comments:
    aid = comment["article_id"]
    ci = comment["comment_index"]
    dg = comment.get("dependency_graph") or []
    if len(dg) < 2:
        continue

    G = nx.DiGraph()
    for node in dg:
        G.add_node(node["id"], tag=node.get("comment_tag", "unknown"),
                   group=to_group(node.get("comment_tag")))
    for node in dg:
        for dep in node.get("depends_on", []):
            G.add_edge(dep["id"], node["id"], edge_type=dep.get("edge_type", "unknown"))

    n = G.number_of_nodes()
    m = G.number_of_edges()
    max_edges = n * (n - 1)
    density = m / max_edges if max_edges > 0 else 0
    avg_in = m / n if n > 0 else 0

    pr = nx.pagerank(G, alpha=0.85) if n > 0 else {}
    bc = nx.betweenness_centrality(G) if n > 1 else {}

    graph_metrics.append({
        "article_id": aid,
        "comment_index": ci,
        "n_nodes": n,
        "n_edges": m,
        "density": density,
        "avg_degree": avg_in,
        "max_in_degree": max(dict(G.in_degree()).values()) if n else 0,
        "max_pagerank": max(pr.values()) if pr else 0,
        "mean_betweenness": np.mean(list(bc.values())) if bc else 0,
    })

    for nid in G.nodes:
        attrs = G.nodes[nid]
        nodes.loc[
            (nodes["article_id"] == aid) &
            (nodes["comment_index"] == ci) &
            (nodes["node_id"] == nid),
            "pagerank"
        ] = pr.get(nid, 0)
        nodes.loc[
            (nodes["article_id"] == aid) &
            (nodes["comment_index"] == ci) &
            (nodes["node_id"] == nid),
            "betweenness"
        ] = bc.get(nid, 0)

gm = pd.DataFrame(graph_metrics)
print(f"Computed graph metrics for {len(gm)} comments (≥2 nodes)")
print()
print(gm[["density", "avg_degree", "n_nodes", "n_edges"]].describe().round(3))

Computed graph metrics for 743 comments (≥2 nodes)

       density  avg_degree  n_nodes  n_edges
count  743.000     743.000  743.000  743.000
mean     0.097       1.022   14.767   15.762
std      0.072       0.250    6.598    8.681
min      0.022       0.167    2.000    1.000
25%      0.058       0.875   10.000    9.000
50%      0.077       1.000   14.000   15.000
75%      0.109       1.174   19.000   21.000
max      0.500       1.929   41.000   49.000


In [16]:
# ── Q4b: Density distribution by article ────────────────────────────────
chart_q4b = (
    alt.Chart(gm, title="Graph density per comment")
    .mark_boxplot(extent="min-max", size=30)
    .encode(
        x=alt.X("article_id:N", title="Article", sort=ARTICLE_IDS),
        y=alt.Y("density:Q", title="Density (edges / possible edges)"),
        color=alt.Color("article_id:N", legend=None),
    )
    .properties(width=400, height=300)
)
chart_q4b

alt.Chart(...)

In [17]:
# ── Q4c: Mean in-degree by category — "business end of the arrow" ───────
in_deg_by_tag = (
    nodes[nodes["group"] != "unknown"]
    .groupby("tag")["in_degree"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values("mean", ascending=False)
)
in_deg_by_tag["se"] = in_deg_by_tag["std"] / np.sqrt(in_deg_by_tag["count"])
in_deg_by_tag["lo"] = in_deg_by_tag["mean"] - 1.96 * in_deg_by_tag["se"]
in_deg_by_tag["hi"] = in_deg_by_tag["mean"] + 1.96 * in_deg_by_tag["se"]

bars = (
    alt.Chart(in_deg_by_tag)
    .mark_bar()
    .encode(
        x=alt.X("mean:Q", title="Mean in-degree (95% CI)"),
        y=alt.Y("tag:N", sort=alt.EncodingSortField(field="mean", order="descending"), title=""),
        color=alt.Color(
            "tag:N",
            legend=None,
        ),
        tooltip=["tag:N", alt.Tooltip("mean:Q", format=".2f"), "count:Q"],
    )
)
error = (
    alt.Chart(in_deg_by_tag)
    .mark_errorbar()
    .encode(
        x="lo:Q", x2="hi:Q",
        y=alt.Y("tag:N", sort=alt.EncodingSortField(field="mean", order="descending")),
    )
)
chart_q4c = (bars + error).properties(
    width=500, height=350,
    title="Mean in-degree by category (who receives arrows = 'business end')"
)
chart_q4c

alt.LayerChart(...)

In [52]:
# ── Q4d: Mean PageRank centrality by group ──────────────────────────────
pr_by_group = (
    nodes.dropna(subset=["pagerank"])
    .groupby("group")["pagerank"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values("mean", ascending=False)
)
pr_by_group = pr_by_group[pr_by_group["group"] != "unknown"]

chart_q4d = (
    alt.Chart(pr_by_group, title="Mean PageRank centrality by functional group")
    .mark_bar()
    .encode(
        x=alt.X("mean:Q", title="Mean PageRank"),
        y=alt.Y("group:N", sort=alt.EncodingSortField(field="mean", order="descending"), title=""),
        color=alt.Color(
            "group:N",
            sort=GROUPS,
            scale=alt.Scale(domain=GROUPS, range=["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]),
            legend=None,
        ),
        tooltip=["group:N", alt.Tooltip("mean:Q", format=".4f"), "count:Q"],
    )
    .properties(width=450, height=200)
)
chart_q4d

KeyError: ['pagerank']

In [54]:
# ── Q4e: Average number of edges from one group to another ──────────────
# For each (src_group, tgt_group), show avg num of edges *per comment* (not just raw total).
edge_enriched = edges.copy()
node_lookup = nodes.set_index(["article_id", "comment_index", "node_id"])[["tag", "group"]]

edge_enriched = edge_enriched.merge(
    node_lookup.rename(columns={"tag": "src_tag", "group": "src_group"}),
    left_on=["article_id", "comment_index", "source"],
    right_index=True,
    how="left",
).merge(
    node_lookup.rename(columns={"tag": "tgt_tag", "group": "tgt_group"}),
    left_on=["article_id", "comment_index", "target"],
    right_index=True,
    how="left",
)

# Count number of comments (article_id, comment_index) pairs overall
n_comments = (
    edge_enriched[["article_id", "comment_index"]]
    .drop_duplicates()
    .shape[0]
)

# Alternatively, for even more precise normalisation: for each (src_group, tgt_group), how many comments *contain at least one such edge*? 
# Here we do simple normalisation by total number of comments.

flow_sum = (
    edge_enriched.groupby(["src_group", "tgt_group"])
    .size()
    .reset_index(name="total_edges")
)
flow_sum = flow_sum[(flow_sum["src_group"] != "unknown") & (flow_sum["tgt_group"] != "unknown")]

flow_sum["avg_edges_per_comment"] = flow_sum["total_edges"] / n_comments

chart_q4e_avg = (
    alt.Chart(flow_sum, title="Avg. edges per comment: source group → target group")
    .mark_rect()
    .encode(
        x=alt.X("tgt_group:N", title="Target (receives arrow)", sort=GROUPS),
        y=alt.Y("src_group:N", title="Source (sends arrow)", sort=GROUPS),
        color=alt.Color("avg_edges_per_comment:Q", title="Avg edges per comment", scale=alt.Scale(scheme="orangered")),
        tooltip=[
            "src_group:N",
            "tgt_group:N",
            alt.Tooltip("total_edges:Q", title="Total edges"),
            alt.Tooltip("avg_edges_per_comment:Q", format=".3f", title="Avg per comment"),
        ],
    )
    .properties(width=350, height=300)
)
chart_q4e_avg

alt.Chart(...)

## Q5 — Sequence statistics: transition chains

Treating each comment as a **sequence** of category labels (in discourse order), what are the most common bigram and trigram transitions? What are the longest recurring chain patterns?

In [59]:
# ── Q5a: Build per-comment group sequences ──────────────────────────────
sequences: list[list[str]] = []
for comment in all_comments:
    dg = comment.get("dependency_graph") or []
    if len(dg) < 2:
        continue
    seq = [to_group(node.get("comment_tag")) for node in sorted(dg, key=lambda n: n["id"])]
    sequences.append(seq)

# Tag-level sequences too
tag_sequences: list[list[str]] = []
for comment in all_comments:
    dg = comment.get("dependency_graph") or []
    if len(dg) < 2:
        continue
    seq = [node.get("comment_tag", "unknown") for node in sorted(dg, key=lambda n: n["id"])]
    tag_sequences.append(seq)

print(f"Built {len(sequences)} sequences (comments with ≥2 sentences)")

Built 873 sequences (comments with ≥2 sentences)


In [60]:
# ── Q5b: Transition matrix (group-level bigrams) ───────────────────────
bigram_counts: Counter = Counter()
for seq in sequences:
    for a, b in zip(seq, seq[1:]):
        bigram_counts[(a, b)] += 1

valid_groups = [g for g in GROUPS]
trans_matrix = pd.DataFrame(0.0, index=valid_groups, columns=valid_groups)
for (a, b), c in bigram_counts.items():
    if a in valid_groups and b in valid_groups:
        trans_matrix.loc[a, b] = c

# Row-normalise to get transition probabilities
row_sums = trans_matrix.sum(axis=1).replace(0, 1)
trans_prob = trans_matrix.div(row_sums, axis=0)

# Melt for Altair heatmap
tm_long = trans_prob.reset_index().melt(id_vars="index", var_name="to", value_name="probability")
tm_long.rename(columns={"index": "from"}, inplace=True)

chart_q5b = (
    alt.Chart(tm_long, title="Group transition probabilities (row-normalised)")
    .mark_rect()
    .encode(
        x=alt.X("to:N", title="Next group", sort=GROUPS),
        y=alt.Y("from:N", title="Current group", sort=GROUPS),
        color=alt.Color("probability:Q", title="P(next | current)", scale=alt.Scale(scheme="blues")),
        tooltip=["from:N", "to:N", alt.Tooltip("probability:Q", format=".2f")],
    )
    .properties(width=400, height=300)
)

# Add text labels
text = (
    alt.Chart(tm_long)
    .mark_text(fontSize=11)
    .encode(
        x=alt.X("to:N", sort=GROUPS),
        y=alt.Y("from:N", sort=GROUPS),
        text=alt.Text("probability:Q", format=".2f"),
        color=alt.condition(
            alt.datum.probability > 0.4,
            alt.value("white"),
            alt.value("black"),
        ),
    )
)
chart_q5b + text

alt.LayerChart(...)

In [61]:
# ── Q5c: Top bigrams and trigrams (fine-grained tag level) ──────────────
tag_bigrams: Counter = Counter()
tag_trigrams: Counter = Counter()

for seq in tag_sequences:
    for a, b in zip(seq, seq[1:]):
        tag_bigrams[(a, b)] += 1
    for a, b, c in zip(seq, seq[1:], seq[2:]):
        tag_trigrams[(a, b, c)] += 1

print("=== Top 15 tag-level BIGRAMS ===")
bi_df = pd.DataFrame(
    [{"from": a, "to": b, "count": c} for (a, b), c in tag_bigrams.most_common(15)]
)
print(bi_df.to_string(index=False))
print()

print("=== Top 15 tag-level TRIGRAMS ===")
tri_df = pd.DataFrame(
    [{"step_1": a, "step_2": b, "step_3": c, "count": n}
     for (a, b, c), n in tag_trigrams.most_common(15)]
)
print(tri_df.to_string(index=False))

=== Top 15 tag-level BIGRAMS ===
                                 from                                     to  count
       L3: Trend and pattern analysis         L3: Trend and pattern analysis   1017
                 Background knowledge                   Background knowledge    750
          Personal/episodic retrieval            Personal/episodic retrieval    742
         Information need / curiosity           Information need / curiosity    581
                      Meta / paratext                        Meta / paratext    540
               Visual data extraction                 Visual data extraction    497
       L3: Trend and pattern analysis           Information need / curiosity    267
                Explanatory inference                  Explanatory inference    261
         Information need / curiosity            Personal/episodic retrieval    261
Predictive / counterfactual inference  Predictive / counterfactual inference    214
      Evaluative / affective judgment      

In [62]:
# ── Q5d: Top bigrams visualised as a bar chart ─────────────────────────
bi_df_top = pd.DataFrame(
    [{"transition": f"{a} → {b}", "count": c}
     for (a, b), c in tag_bigrams.most_common(20)]
)

chart_q5d = (
    alt.Chart(bi_df_top, title="Top 20 category bigrams (sequential transitions)")
    .mark_bar()
    .encode(
        x=alt.X("count:Q", title="Count"),
        y=alt.Y("transition:N", sort="-x", title=""),
        tooltip=["transition:N", "count:Q"],
    )
    .properties(width=500, height=450)
)
chart_q5d

alt.Chart(...)

In [63]:
# ── Q5e: Longest recurring chains (maximal repeated substrings) ─────────
from collections import defaultdict as ddict

def extract_ngrams(seq, n):
    return [tuple(seq[i:i+n]) for i in range(len(seq) - n + 1)]

# Find the longest n-gram that appears in ≥ 5 distinct comments
group_sequences_tuples = [tuple(s) for s in sequences]

best_n = 0
best_chains = []
for n in range(2, 12):
    ngram_counts: Counter = Counter()
    for seq in group_sequences_tuples:
        seen = set()
        for ng in extract_ngrams(list(seq), n):
            if ng not in seen:
                ngram_counts[ng] += 1
                seen.add(ng)
    frequent = [(ng, c) for ng, c in ngram_counts.items() if c >= 5]
    if not frequent:
        break
    best_n = n
    best_chains = sorted(frequent, key=lambda x: -x[1])

print(f"Longest group-level chain appearing in ≥5 comments: length {best_n}")
print()
chain_rows = []
for chain, count in best_chains[:20]:
    chain_str = " → ".join(chain)
    chain_rows.append({"chain": chain_str, "length": len(chain), "n_comments": count})
    print(f"  [{count:3d} comments]  {chain_str}")

chain_df = pd.DataFrame(chain_rows)

Longest group-level chain appearing in ≥5 comments: length 11

  [ 25 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → hypothetical_reasoning → hypothetical_reasoning → hypothetical_reasoning → hypothetical_reasoning
  [ 24 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation
  [ 21 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → hypothetical_reasoning
  [ 21 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_obse

In [64]:
# ── Q5f: Recurring chains bar chart ─────────────────────────────────────
if not chain_df.empty:
    chart_q5f = (
        alt.Chart(chain_df.head(20), title=f"Most common group chains (length ≥ {best_n})")
        .mark_bar()
        .encode(
            x=alt.X("n_comments:Q", title="Number of comments containing chain"),
            y=alt.Y("chain:N", sort="-x", title=""),
            color=alt.Color("length:O", title="Chain length", scale=alt.Scale(scheme="viridis")),
            tooltip=["chain:N", "length:Q", "n_comments:Q"],
        )
        .properties(width=500, height=min(400, 25 * len(chain_df.head(20))))
    )
    chart_q5f.display()
else:
    print("No recurring chains found.")

alt.Chart(...)

In [66]:
# ── Q5f: Recurring chains bar chart ─────────────────────────────────────
if not chain_df.empty:
    print("Examples of most common group chains (length ≥ {}):\n".format(best_n))
    for idx, row in chain_df.head(20).iterrows():
        print(f"  [{row['n_comments']:3d} comments]  {row['chain']}")
    print()
    chart_q5f = (
        alt.Chart(chain_df.head(20), title=f"Most common group chains (length ≥ {best_n})")
        .mark_bar()
        .encode(
            x=alt.X("n_comments:Q", title="Number of comments containing chain"),
            y=alt.Y("chain:N", sort="-x", title=""),
            color=alt.Color("length:O", title="Chain length", scale=alt.Scale(scheme="viridis")),
            tooltip=["chain:N", "length:Q", "n_comments:Q"],
        )
        .properties(width=500, height=min(400, 25 * len(chain_df.head(20))))
    )
    chart_q5f.display()
else:
    print("No recurring chains found.")

Examples of most common group chains (length ≥ 11):

  [ 25 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → hypothetical_reasoning → hypothetical_reasoning → hypothetical_reasoning → hypothetical_reasoning
  [ 24 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation
  [ 21 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → hypothetical_reasoning
  [ 21 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → 

alt.Chart(...)

In [68]:
# ── Q5f: Recurring chains bar chart ─────────────────────────────────────
if not chain_df.empty:
    print("Examples of most common group chains (length ≥ {}):\n".format(best_n))
    for idx, row in chain_df.head(20).iterrows():
        print(f"  [{row['n_comments']:3d} comments]  {row['chain']}")
        # Print comments where this chain occurs
        # Assume 'comment_indices' in row contains list of indices in the original dataframe (or modify if it's another key)
        comment_locs = row.get('comment_indices') or row.get('comment_idxs') or row.get('indices')
        if comment_locs is not None:
            for i, ci in enumerate(comment_locs[:5]):  # Show up to 5 examples per chain
                try:
                    # Try node_df or another dataframe containing original comments
                    comment_row = node_df.iloc[ci] if 'node_df' in globals() else None
                except Exception:
                    comment_row = None
                if comment_row is not None:
                    print("     - Article {} comment {}: {}".format(
                        comment_row.get('article_id', ''),
                        comment_row.get('comment_index', ''),
                        comment_row.get('sentence', '(sentence unavailable)')
                    ))
                else:
                    print(f"     - Comment index: {ci} (detail unavailable)")
            if len(comment_locs) > 5:
                print(f"       ... ({len(comment_locs)-5} more not shown)")
    print()
    chart_q5f = (
        alt.Chart(chain_df.head(20), title=f"Most common group chains (length ≥ {best_n})")
        .mark_bar()
        .encode(
            x=alt.X("n_comments:Q", title="Number of comments containing chain"),
            y=alt.Y("chain:N", sort="-x", title=""),
            color=alt.Color("length:O", title="Chain length", scale=alt.Scale(scheme="viridis")),
            tooltip=["chain:N", "length:Q", "n_comments:Q"],
        )
        .properties(width=500, height=min(400, 25 * len(chain_df.head(20))))
    )
    chart_q5f.display()
else:
    print("No recurring chains found.")

Examples of most common group chains (length ≥ 11):

  [ 25 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → hypothetical_reasoning → hypothetical_reasoning → hypothetical_reasoning → hypothetical_reasoning
  [ 24 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation
  [ 21 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → hypothetical_reasoning
  [ 21 comments]  visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → visual_observation → 

alt.Chart(...)

In [65]:
# ── Q5g: "Run length" analysis — how long do people stay in each group? ─
run_lengths: list[dict] = []
for seq in sequences:
    current = seq[0]
    length = 1
    for g in seq[1:]:
        if g == current:
            length += 1
        else:
            run_lengths.append({"group": current, "run_length": length})
            current = g
            length = 1
    run_lengths.append({"group": current, "run_length": length})

rl = pd.DataFrame(run_lengths)
rl = rl[rl["group"] != "unknown"]

rl_summary = (
    rl.groupby("group")["run_length"]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
    .round(2)
)
print("Run-length statistics: how many consecutive sentences of the same group")
print(rl_summary)
print()

chart_q5g = (
    alt.Chart(rl, title="Consecutive run lengths by group")
    .mark_boxplot(extent="min-max")
    .encode(
        x=alt.X("group:N", title="Group", sort=GROUPS),
        y=alt.Y("run_length:Q", title="Consecutive sentences in same group"),
        color=alt.Color(
            "group:N", sort=GROUPS, legend=None,
            scale=alt.Scale(domain=GROUPS, range=["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]),
        ),
    )
    .properties(width=400, height=300)
)
chart_q5g

Run-length statistics: how many consecutive sentences of the same group
                        mean  median  max  count
group                                           
visual_observation      2.66     2.0   18   1342
personal_relevance      2.08     2.0   11   1071
hypothetical_reasoning  1.91     1.0   11   1458
background_knowledge    1.78     1.0   12    961
meta                    1.70     1.0   37    773



alt.Chart(...)

In [ ]:
# ── Q5f: Recurring chains bar chart ─────────────────────────────────────
if not chain_df.empty:
    chart_q5f = (
        alt.Chart(chain_df.head(20), title=f"Most common group chains (length ≥ {best_n})")
        .mark_bar()
        .encode(
            x=alt.X("n_comments:Q", title="Number of comments containing chain"),
            y=alt.Y("chain:N", sort="-x", title=""),
            color=alt.Color("length:O", title="Chain length", scale=alt.Scale(scheme="viridis")),
            tooltip=["chain:N", "length:Q", "n_comments:Q"],
        )
        .properties(width=500, height=min(400, 25 * len(chain_df.head(20))))
    )
    chart_q5f.display()
else:
    print("No recurring chains found.")